# 01. 고객 이탈 예측 — 로지스틱 회귀 vs SVM 분류 실습
> SK네트웍스 Family AI 캠프 37기 · 2026-10-15

**목표**: 통신사 고객 정보로 **"어떤 고객이 떠날(Churn) 가능성이 높은가?"**를 예측하는 이진 분류 모델을 만든다.

| 항목 | 내용 |
|---|---|
| 데이터 | Kaggle Telco Customer Churn (`Telco-Customer-Churn.csv`) |
| 규모 | 7,043행 × 21열 (TotalCharges에 공백 문자열 11건) |
| 정답(타겟) | `Churn` — Yes(이탈) / No(유지) |
| 모델 | LogisticRegression, SVC(SVM) |

**진행 순서**: 데이터 불러오기 → 전처리 → EDA(시각화) → 인코딩 → 학습/평가 데이터 분리 → 스케일링 → 모델 학습·평가 → 비교

## 1. 데이터 불러오기

In [1]:
# 필요한 라이브러리 임포트
import pandas as pd
import plotly.express as px

# 경고 메시지 숨기기
import warnings
warnings.filterwarnings('ignore')

In [2]:
# 파일 경로 설정 -> 노트북과 같은 폴더에 CSV를 두고 상대경로로 읽기
file_path = './Telco-Customer-Churn.csv'

# DataFrame 생성
df = pd.read_csv(file_path)

# 결과 확인
print(f'데이터 크기(행, 열): {df.shape}')
display(df.head())

데이터 크기(행, 열): (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### 컬럼 설명 (데이터 설명서 요약)
| 구분 | 컬럼 |
|---|---|
| 고객 인구통계 | `customerID`(학습 제외), `gender`, `SeniorCitizen`(1=65세 이상), `Partner`, `Dependents` |
| 가입 서비스 | `PhoneService`, `MultipleLines`, `InternetService`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies` |
| 계약·결제 | `tenure`(가입 개월 수), `Contract`, `PaperlessBilling`, `PaymentMethod`, `MonthlyCharges`(월 요금), `TotalCharges`(총 누적 요금) |
| 정답 | `Churn` (Yes / No) |

## 2. 데이터 전처리

In [3]:
# 기본 정보 확인(행·열 수, 결측치, 자료형) -> info()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

👉 `TotalCharges`는 요금(숫자)인데 자료형이 **문자열**이다. → 숫자로 바꿀 수 없는 값이 섞여 있다는 뜻.

In [4]:
# 문제점: TotalCharges 컬럼에 숫자로 변환할 수 없는 공백 문자열(' ')이 포함됨

## TotalCharges 값이 공백(' ')인 행의 인덱스 추출
space_index = df.query("TotalCharges == ' '").index
print(f'공백 행 인덱스: {space_index.tolist()}')
print(f'공백 행 개수: {len(space_index)}')

print('-'*80)

## 공백 행 확인 -> 모두 tenure(가입 기간)가 0 = 아직 첫 요금이 청구되지 않은 신규 고객
display(df.loc[space_index, ['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']])

공백 행 인덱스: [488, 753, 936, 1082, 1340, 3331, 3826, 4380, 5218, 6670, 6754]
공백 행 개수: 11
--------------------------------------------------------------------------------


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


In [5]:
# 전처리 1단계: TotalCharges 자료형 변환(문자열 -> 숫자) -> pd.to_numeric()
## errors='coerce': 변환할 수 없는 값(' ')은 NaN(결측치)으로 바꿈
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print(f"TotalCharges 자료형: {df['TotalCharges'].dtype}")
print(f"TotalCharges 결측치 개수: {df['TotalCharges'].isna().sum()}")

TotalCharges 자료형: float64
TotalCharges 결측치 개수: 11


In [6]:
# 전처리 2단계: 결측치가 있는 행 제거 -> dropna()
## ignore_index=True: 제거 후 행 인덱스를 0부터 다시 매김
df1 = df.dropna(ignore_index=True)

print(f'제거 전: {df.shape} -> 제거 후: {df1.shape}')

제거 전: (7043, 21) -> 제거 후: (7032, 21)


> 💡 대안: 11건(0.16%)뿐이라 제거해도 영향이 작다. 제거 대신 `tenure=0`이므로 `TotalCharges`를 **0으로 채우는** 방법(`fillna(0)`)도 가능하다.

## 3. 데이터 탐색 및 EDA

In [7]:
# 분석 목표: 정답 컬럼(Churn)의 항목별 비율 -> 원 그래프

## 전체 고객 중 이탈자의 비율 -> value_counts(normalize=True) -> reset_index()
df_churn_ratio = df1['Churn'].value_counts(normalize=True).reset_index()
display(df_churn_ratio)

## 시각화
fig1 = px.pie(
    data_frame=df_churn_ratio,
    names='Churn',
    values='proportion',
    title='고객 이탈 여부 비중',
    hole=0.3
)
fig1.show()

,Churn,proportion
0,No,0.734215
1,Yes,0.265785


👉 유지(No) 약 73%, 이탈(Yes) 약 27% → **불균형 데이터**.
아무것도 학습하지 않고 전부 "No"라고 찍어도 정확도가 약 0.73이 나온다. → 정확도만 보면 안 되고 **이탈 고객을 얼마나 잡아내는지(재현율)**도 함께 봐야 한다.

In [8]:
# 분석 목표: 계약 형태(Contract)와 고객 이탈의 관계 -> 막대 그래프

## Contract, Churn 기준으로 그룹화 -> 그룹별 행의 수(size) -> reset_index()
df_contract_churn_count = df1.groupby(by=['Contract', 'Churn']).size().reset_index(name='Count')
display(df_contract_churn_count)

## 시각화
fig2 = px.bar(
    data_frame=df_contract_churn_count,
    x='Contract',
    y='Count',
    color='Churn',
    barmode='group',
    text_auto=True,
    title='계약 형태별 이탈/유지 고객 수'
)
fig2.show()

,Contract,Churn,Count
0,Month-to-month,No,2220
1,Month-to-month,Yes,1655
2,One year,No,1306
3,One year,Yes,166
4,Two year,No,1637
5,Two year,Yes,48


In [9]:
# 보충: 계약 형태별 이탈률(%) -> 개수보다 비율로 보면 차이가 더 분명함
contract_churn_rate = (
    df1.groupby('Contract')['Churn']
       .apply(lambda s: (s == 'Yes').mean() * 100)
       .round(1)
       .reset_index(name='이탈률(%)')
)
display(contract_churn_rate)

,Contract,이탈률(%)
0,Month-to-month,42.7
1,One year,11.3
2,Two year,2.8


👉 **월 단위 계약(Month-to-month)** 고객의 이탈률이 압도적으로 높고, 1년·2년 약정 고객은 거의 이탈하지 않는다.

In [10]:
# 분석 목표: 수치형 컬럼(tenure, MonthlyCharges, TotalCharges)과 고객 이탈의 관계 -> 히스토그램

## 컬럼별 그래프 제목
hist_titles = {
    'tenure': '가입 기간(개월)에 따른 이탈 분포',
    'MonthlyCharges': '월 사용 요금과 고객 이탈의 관계',
    'TotalCharges': '총 누적 요금과 고객 이탈의 관계',
}

## 같은 형태의 그래프 3개 -> 반복문으로 한 번에 그리기
for col, title in hist_titles.items():
    fig = px.histogram(
        data_frame=df1,
        x=col,
        color='Churn',
        barmode='group',
        title=title
    )
    # 모든 막대에 검은색 경계선 넣기
    fig.update_traces(marker_line_color='black', marker_line_width=1)
    fig.show()

| 컬럼 | 관찰 |
|---|---|
| `tenure` | 가입 초기(0~10개월) 고객의 이탈이 많고, 오래된 고객일수록 이탈이 적다 |
| `MonthlyCharges` | 월 요금이 높은 구간(약 70~105)에서 이탈이 많다 |
| `TotalCharges` | 누적 요금이 낮은(= 가입 기간이 짧은) 고객의 이탈이 많다 |

## 4. 인코딩

In [11]:
# 불필요한 컬럼(고객 ID) 삭제 -> 고객마다 다른 고유 번호라 예측에 도움이 안 됨
df2 = df1.drop(columns=['customerID'])

# 정답 컬럼(Churn) -> 라벨 인코딩 ('No':0, 'Yes':1) -> map()
df2['Churn'] = df2['Churn'].map({'No': 0, 'Yes': 1})

display(df2.head())

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


In [12]:
# feature의 범주형 컬럼 -> 원-핫 인코딩

## 범주형(문자열) 컬럼 이름 리스트 추출 -> select_dtypes() -> columns -> tolist()
## (pandas 3.x부터 문자열 자료형이 'str'이므로 'object'와 'string'을 함께 지정)
categorical_cols_list = df2.select_dtypes(include=['object', 'string']).columns.tolist()
print(f'범주형 컬럼({len(categorical_cols_list)}개): \n{categorical_cols_list}')

print('-'*80)

## 원-핫 인코딩 -> pd.get_dummies(), dtype=int로 0/1 정수 변환
df_onehot = pd.get_dummies(data=df2, columns=categorical_cols_list, dtype=int)

print(f'인코딩 전: {df2.shape} -> 인코딩 후: {df_onehot.shape}')
display(df_onehot.head())

범주형 컬럼(15개): 
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
--------------------------------------------------------------------------------
인코딩 전: (7032, 20) -> 인코딩 후: (7032, 46)


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,gender_Female,gender_Male,Partner_No,Partner_Yes,Dependents_No,...,StreamingMovies_Yes,Contract_Month-to-month,Contract_One year,Contract_Two year,PaperlessBilling_No,PaperlessBilling_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,1,0,0,1,1,...,0,1,0,0,0,1,0,0,1,0
1,0,34,56.95,1889.50,0,0,1,1,0,1,...,0,0,1,0,1,0,0,0,0,1
2,0,2,53.85,108.15,1,0,1,1,0,1,...,0,1,0,0,0,1,0,0,0,1
3,0,45,42.30,1840.75,0,0,1,1,0,1,...,0,0,1,0,1,0,1,0,0,0
4,0,2,70.70,151.65,1,1,0,1,0,1,...,0,1,0,0,0,1,0,0,1,0


## 5. 학습용 데이터와 평가용 데이터 생성

In [13]:
# X_data(문제) -> 정답 컬럼 제거
X_data = df_onehot.drop(columns=['Churn'])

# y_data(정답) -> 정답 컬럼만 선택
y_data = df_onehot['Churn']

print(f'X_data: {X_data.shape}, y_data: {y_data.shape}')
print(f'정답의 항목별 비율: \n{y_data.value_counts(normalize=True)}')

X_data: (7032, 45), y_data: (7032,)
정답의 항목별 비율: 
Churn
0    0.734215
1    0.265785
Name: proportion, dtype: float64


In [14]:
# 80:20 비율로 학습용/평가용 데이터 분리
from sklearn.model_selection import train_test_split

## random_state: 무작위성 고정(재현성), stratify: 정답 비율을 학습/평가에 동일하게 유지
X_train, X_test, y_train, y_test = train_test_split(
    X_data,
    y_data,
    test_size=0.2,
    random_state=0,
    stratify=y_data
)

print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')

X_train: (5625, 45), X_test: (1407, 45)


In [15]:
# stratify 효과 확인 -> 전체·학습용·평가용의 정답 비율이 거의 같아야 함
ratio_df = pd.DataFrame({
    '전체': y_data.value_counts(normalize=True),
    '학습용': y_train.value_counts(normalize=True),
    '평가용': y_test.value_counts(normalize=True),
}).round(4)
display(ratio_df)

,전체,학습용,평가용
Churn,,,
0,0.7342,0.7342,0.7342
1,0.2658,0.2658,0.2658


## 6. 스케일링(표준화)
`tenure`(0~72), `MonthlyCharges`(18~119), `TotalCharges`(18~8,685)와 0/1 원-핫 컬럼의 **값 범위가 크게 다르다.**
- **SVM**: 거리(마진) 기반이라 큰 값의 컬럼이 결과를 좌우함 → 스케일링이 사실상 필수
- **로지스틱 회귀**: 스케일링하지 않으면 학습이 수렴하지 않을 수 있음(`ConvergenceWarning`)

⚠️ `fit`은 **학습용 데이터로만** 하고, 평가용 데이터에는 `transform`만 적용한다. (평가 데이터 정보가 학습에 새어 들어가는 것 방지)

In [16]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # 학습용: 평균·표준편차 계산 + 변환
X_test_scaled = scaler.transform(X_test)         # 평가용: 학습용 기준으로 변환만

print(f'스케일링 후 학습용 평균(앞 4개 컬럼): {X_train_scaled[:, :4].mean(axis=0).round(3)}')
print(f'스케일링 후 학습용 표준편차(앞 4개 컬럼): {X_train_scaled[:, :4].std(axis=0).round(3)}')

스케일링 후 학습용 평균(앞 4개 컬럼): [ 0.  0.  0. -0.]
스케일링 후 학습용 표준편차(앞 4개 컬럼): [1. 1. 1. 1.]


## 7. 평가 함수 준비
불균형 데이터이므로 정확도와 함께 **혼동 행렬·정밀도·재현율·F1**을 확인한다.

| 지표 | 의미 (이탈=1 기준) |
|---|---|
| 정확도(accuracy) | 전체 중 맞힌 비율 |
| 정밀도(precision) | 이탈이라고 예측한 고객 중 실제 이탈 비율 |
| 재현율(recall) | 실제 이탈 고객 중 모델이 잡아낸 비율 ← 이탈 방어 마케팅에서 중요 |
| F1 | 정밀도와 재현율의 조화 평균 |

In [17]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

def evaluate(model_name, y_true, y_pred):
    """정확도 + 혼동 행렬 + 분류 리포트를 출력하는 함수"""
    acc = accuracy_score(y_true, y_pred)
    print(f'[{model_name}] 정확도: {acc:.4f}')
    print('-'*80)

    cm = pd.DataFrame(
        confusion_matrix(y_true, y_pred),
        index=['실제 유지(0)', '실제 이탈(1)'],
        columns=['예측 유지(0)', '예측 이탈(1)']
    )
    display(cm)
    print('-'*80)

    print(classification_report(y_true, y_pred, target_names=['유지(0)', '이탈(1)'], digits=4))
    return acc

## 8. LogisticRegression(로지스틱 회귀) 모델

### 모델 생성

In [18]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(random_state=0)

### 모델 학습: fit()

In [19]:
lr.fit(X_train_scaled, y_train)
print('학습 완료')

학습 완료


### 평가용 데이터를 이용한 예측: predict()

In [20]:
pred_test_lr = lr.predict(X_test_scaled)
print(pred_test_lr[:20])

[1 0 0 1 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 1]


### 모델 평가

In [21]:
acc_lr = evaluate('LogisticRegression', y_test, pred_test_lr)

[LogisticRegression] 정확도: 0.8081
--------------------------------------------------------------------------------


,예측 유지(0),예측 이탈(1)
실제 유지(0),930,103
실제 이탈(1),167,207


--------------------------------------------------------------------------------
              precision    recall  f1-score   support

       유지(0)     0.8478    0.9003    0.8732      1033
       이탈(1)     0.6677    0.5535    0.6053       374

    accuracy                         0.8081      1407
   macro avg     0.7578    0.7269    0.7393      1407
weighted avg     0.7999    0.8081    0.8020      1407



In [22]:
# 보충: 로지스틱 회귀의 계수(coef_) -> 어떤 특성이 이탈 확률을 높이고(+) 낮추는지(-) 확인
## 스케일링된 데이터로 학습했으므로 계수 크기끼리 비교 가능
coef_df = pd.DataFrame({'feature': X_train.columns, 'coef': lr.coef_[0]}).sort_values('coef')

print('이탈 확률을 낮추는 특성 Top 5')
display(coef_df.head(5))
print('이탈 확률을 높이는 특성 Top 5')
display(coef_df.tail(5).iloc[::-1])

이탈 확률을 낮추는 특성 Top 5


,feature,coef
1,tenure,-1.477594
38,Contract_Two year,-0.273035
15,InternetService_DSL,-0.142961
29,TechSupport_Yes,-0.095110
20,OnlineSecurity_Yes,-0.086793


이탈 확률을 높이는 특성 Top 5


,feature,coef
3,TotalCharges,0.776934
36,Contract_Month-to-month,0.284506
16,InternetService_Fiber optic,0.163939
43,PaymentMethod_Electronic check,0.130039
27,TechSupport_No,0.112978


👉 `tenure`(가입 기간)가 길수록, 2년 약정일수록 이탈 확률이 낮아진다. 월 단위 계약·광섬유 인터넷·전자 수표 결제는 이탈 확률을 높인다.

⚠️ `TotalCharges`는 대략 `tenure × MonthlyCharges`라 `tenure`와 강하게 연관(다중공선성)되어 있다. 그래서 계수의 부호를 단독으로 해석하면 안 된다(EDA에서는 누적 요금이 낮을수록 이탈이 많았음).

## 9. SVM 분류 모델
**SVM(Support Vector Machine)**: 두 클래스 사이의 **마진(경계와 가장 가까운 데이터 사이 거리)을 최대화**하는 결정 경계를 찾는 모델.
`SVC`의 기본 커널은 `rbf`(비선형 경계 가능), 기본 `C=1.0`.

### 모델 생성

In [23]:
from sklearn.svm import SVC

svm = SVC(random_state=0)

### 모델 학습: fit()

In [24]:
svm.fit(X_train_scaled, y_train)
print('학습 완료')

학습 완료


### 평가용 데이터를 이용한 예측: predict()

In [25]:
pred_test_svm = svm.predict(X_test_scaled)
print(pred_test_svm[:20])

[1 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 1]


### 모델 평가

In [26]:
acc_svm = evaluate('SVC', y_test, pred_test_svm)

[SVC] 정확도: 0.7974
--------------------------------------------------------------------------------


,예측 유지(0),예측 이탈(1)
실제 유지(0),946,87
실제 이탈(1),198,176


--------------------------------------------------------------------------------
              precision    recall  f1-score   support

       유지(0)     0.8269    0.9158    0.8691      1033
       이탈(1)     0.6692    0.4706    0.5526       374

    accuracy                         0.7974      1407
   macro avg     0.7481    0.6932    0.7108      1407
weighted avg     0.7850    0.7974    0.7850      1407



## 10. 모델 비교
기준선(baseline): 모든 고객을 "유지(0)"로 예측했을 때의 정확도

In [27]:
from sklearn.metrics import precision_score, recall_score, f1_score

baseline_acc = (y_test == 0).mean()

result_df = pd.DataFrame({
    '모델': ['Baseline(전부 유지 예측)', 'LogisticRegression', 'SVC'],
    '정확도': [baseline_acc, acc_lr, acc_svm],
    '정밀도(이탈)': [0, precision_score(y_test, pred_test_lr), precision_score(y_test, pred_test_svm)],
    '재현율(이탈)': [0, recall_score(y_test, pred_test_lr), recall_score(y_test, pred_test_svm)],
    'F1(이탈)': [0, f1_score(y_test, pred_test_lr), f1_score(y_test, pred_test_svm)],
}).round(4)
display(result_df)

,모델,정확도,정밀도(이탈),재현율(이탈),F1(이탈)
0,Baseline(전부 유지 예측),0.7342,0.0000,0.0000,0.0000
1,LogisticRegression,0.8081,0.6677,0.5535,0.6053
2,SVC,0.7974,0.6692,0.4706,0.5526


## 11. 정리 ✅
- `TotalCharges`의 공백 문자열 11건(가입 0개월 신규 고객) → `pd.to_numeric(errors='coerce')` + `dropna()`로 처리
- EDA: **월 단위 계약, 짧은 가입 기간, 높은 월 요금** 고객의 이탈이 많다
- 범주형 15개 컬럼 원-핫 인코딩 → 특성 45개
- 정답 비율이 73:27인 불균형 데이터 → `stratify`로 분리, 정확도 외에 **재현율·F1**도 확인
- 거리 기반 SVM은 **스케일링 필수**, 스케일러는 학습용으로만 `fit`
- 결과: LogisticRegression 정확도 **0.808**, SVC **0.797** → 둘 다 기준선(0.734)보다 높음
- 하지만 이탈 재현율은 LR 0.55, SVC 0.47 → 실제 이탈 고객의 절반 정도만 잡아냄
- 이 데이터에서는 단순한 로지스틱 회귀가 SVM보다 정확도·재현율 모두 조금 더 좋음

**더 해볼 것**
- `class_weight='balanced'`로 이탈 클래스에 가중치 → 재현율 향상
- `predict_proba()` + 임계값(threshold) 조정
- SVM의 `C`, `gamma`, `kernel` 하이퍼파라미터 튜닝(`GridSearchCV`)